In [ ]:
import json
import uuid
from datetime import datetime, timezone
from typing import Any

from dbx_tools.graphiti.runtime import GraphitiRuntime
from graphiti_core.nodes import EpisodeType, EpisodicNode

"""Write and immediately verify one episode through GraphitiRuntime."""

dbutils: Any = globals()["dbutils"]

DATABASE_URL = "projects/lfp-chat-db/branches/production/endpoints/primary"
MARKER = f"dbx-tools-runtime-persistence-{uuid.uuid4()}"


def _episode_record(episode):
    """Return JSON-safe episode fields used by the task result."""
    return {
        "uuid": str(episode.uuid),
        "name": episode.name,
        "group_id": episode.group_id,
        "content": episode.content,
        "source_description": episode.source_description,
        "source": getattr(episode.source, "value", episode.source),
        "valid_at": episode.valid_at.isoformat() if episode.valid_at else None,
        "created_at": episode.created_at.isoformat() if episode.created_at else None,
    }


async def _write_memory():
    """Persist an episode and verify direct and grouped reads before shutdown."""
    runtime = GraphitiRuntime.from_options({"databaseUrl": DATABASE_URL})
    await runtime.start()
    try:
        added = await runtime.graphiti.add_episode(
            name="dbx-tools serverless persistence validation",
            episode_body=f"Persistent marker: {MARKER}",
            source_description="dbx-tools GraphitiRuntime validation",
            source=EpisodeType.text,
            group_id="main",
            reference_time=datetime.now(timezone.utc),
        )
        episode_uuid = str(added.episode.uuid)
        direct = await EpisodicNode.get_by_uuid(runtime.graphiti.driver, episode_uuid)
        grouped = await EpisodicNode.get_by_group_ids(
            runtime.graphiti.driver,
            ["main"],
            limit=100,
        )
        grouped_matches = [episode for episode in grouped if MARKER in episode.content]
        if MARKER not in direct.content or not grouped_matches:
            raise AssertionError(
                f"GraphitiRuntime did not persist episode {episode_uuid} with marker {MARKER}"
            )
        return {
            "marker": MARKER,
            "episode_uuid": episode_uuid,
            "write_verified": True,
            "direct": _episode_record(direct),
            "grouped_match_count": len(grouped_matches),
        }
    finally:
        await runtime.close()


result = await _write_memory()
dbutils.jobs.taskValues.set(key="marker", value=result["marker"])
dbutils.jobs.taskValues.set(key="episode_uuid", value=result["episode_uuid"])
dbutils.notebook.exit(json.dumps(result))